# Decoding in Autoregressive LLMs
### DS 207 — Introduction to NLP | IISc Bangalore | April 2026
#### Guest Lecture: Apoorv Saxena, Inception AI

---

This notebook implements key decoding strategies **from scratch** using GPT-2.

**What we'll cover:**
1. How an LLM produces text — logits → probabilities
2. Greedy decoding (and why it fails)
3. Temperature sampling
4. Top-k sampling
5. Top-p (nucleus) sampling
6. Min-p sampling
7. The autoregressive speed bottleneck
8. [Demo] Speculative decoding — draft + verify


## Setup
We use **GPT-2** (117M parameters) throughout this notebook — small enough to run on CPU.
If you have a GPU, it will be used automatically.


In [ ]:
# If running for the first time, uncomment:
# !pip install transformers torch matplotlib --quiet

from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import time

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 12


In [ ]:
model_name = 'gpt2-medium'   # ~117M params — change to 'gpt2-large' for better quality

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model     = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

VOCAB_SIZE = len(tokenizer)
print(f'Model: {model_name}  |  Device: {device}  |  Vocab size: {VOCAB_SIZE:,}')


---
## Part 1 — How Does an LLM Produce Text?

At every generation step, the model takes the current sequence of tokens and outputs a
**logit** vector of size `vocab_size`. Logits are raw (unnormalised) scores — one per token.

To get a **probability distribution** we apply *softmax*:

$$P(w) = \frac{e^{z_w}}{\sum_v e^{z_v}}$$

Then we **choose** the next token somehow — that choice is the *decoding strategy*.


In [ ]:
prompt = 'The researchers found that the new method'
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = model(input_ids)

# logits shape: [batch, seq_len, vocab_size]  — we want the LAST token
logits = outputs.logits[0, -1, :]   # shape: [vocab_size]

print(f'Input tokens  : {input_ids.shape}  → {input_ids.tolist()[0]}')
print(f'Logits shape  : {logits.shape}')
print(f'Logit range   : [{logits.min():.2f}, {logits.max():.2f}]')


In [ ]:
outputs.logits.shape

In [ ]:
def show_top_tokens(logits, k=15, title='Next-token probabilities'):
    probs = F.softmax(logits, dim=-1)
    top_vals, top_idx = torch.topk(probs, k)
    labels = [repr(tokenizer.decode([i])) for i in top_idx]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(range(k), top_vals.cpu().numpy(), color='steelblue')
    ax.set_xticks(range(k))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel('Probability')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

show_top_tokens(logits, title=f'Next token after: "{prompt}"')


---
## Part 2 — Greedy Decoding

The simplest strategy: **always pick the highest-probability token**.

$$w_t = \arg\max_w P(w \mid w_1, \dots, w_{t-1})$$

**Pros:** Fast, deterministic, reproducible.  
**Cons:** Often repetitive and dull — the model gets 'stuck' in loops.


In [ ]:
def greedy_decode(prompt, max_new_tokens=50, verbose=True):
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    if verbose:
        print(prompt, end='', flush=True)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits      = model(input_ids).logits[0, -1, :]
            next_tok    = torch.argmax(logits).unsqueeze(0).unsqueeze(0)   # shape [1,1]
            input_ids   = torch.cat([input_ids, next_tok], dim=-1)

            if verbose:
                print(tokenizer.decode([next_tok.item()]), end='', flush=True)

    if verbose:
        print()
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


In [ ]:
print('=== Greedy decoding ===\n')
_ = greedy_decode('The cat sat on the', max_new_tokens=60)


**Repetition!** Greedy decoding tends to loop — once a token is the locally best choice
it keeps being chosen, because the context now contains more of itself.

Try a different prompt:


In [ ]:
_ = greedy_decode('You are the', max_new_tokens=50)


---
## Part 3 — Sampling with Temperature

Instead of always picking the top token, we **sample** from the distribution.

**Temperature** $T$ controls how sharp or flat the distribution is:

$$P_T(w) = \text{softmax}\!\left(\frac{z_w}{T}\right)$$

- $T \to 0$: distribution collapses to greedy (peak at argmax)
- $T = 1$: original model distribution
- $T > 1$: flatter — more random, more surprising outputs


In [ ]:
def apply_temperature(logits, temperature=1.0):
    """Scale logits by 1/temperature before softmax."""
    if temperature <= 0:
        raise ValueError('Temperature must be > 0')
    return logits / temperature


In [ ]:
# Visualise how temperature reshapes the distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, T in zip(axes, [0.3, 1.0, 2.0]):
    scaled_logits = apply_temperature(logits, T)
    probs = F.softmax(scaled_logits, dim=-1)
    top_vals, top_idx = torch.topk(probs, 10)
    labels = [repr(tokenizer.decode([i])) for i in top_idx]

    ax.bar(range(10), top_vals.cpu().numpy(), color='tomato')
    ax.set_xticks(range(10))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'T = {T}', fontweight='bold')
    ax.set_ylabel('Probability')

plt.suptitle(f'Effect of temperature on: "{prompt}"', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
def sampling_decode(prompt, max_new_tokens=80, temperature=1.0, verbose=True):
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    if verbose:
        print(prompt, end='', flush=True)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits    = model(input_ids).logits[0, -1, :]
            logits    = apply_temperature(logits, temperature)
            probs     = F.softmax(logits, dim=-1)
            next_tok  = torch.multinomial(probs, 1).unsqueeze(0)   # shape [1,1]
            input_ids = torch.cat([input_ids, next_tok], dim=-1)

            if verbose:
                print(tokenizer.decode([next_tok.item()]), end='', flush=True)

    if verbose:
        print()
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


In [ ]:
prompt2 = 'The researchers found that the new method'
for T in [0.5, 1.0, 1.5]:
    print(f'\n--- Temperature = {T} ---')
    sampling_decode(prompt2, max_new_tokens=60, temperature=T)


---
## Part 4 — Top-k Sampling

Problem with pure sampling: even with $T=1$, tokens with tiny probabilities can occasionally
be sampled — leading to incoherent text.

**Top-k**: restrict sampling to the **k most probable tokens** and renormalise.

Tokens outside the top-k get probability 0 (logit = $-\infty$).

**Intuition:** we always sample from a 'sensible' shortlist.


In [ ]:
def apply_top_k(logits, k=50):
    """Zero out all logits except the top k. Returns modified logits."""
    if k <= 0:
        return logits  # no filtering
    top_k_vals, _ = torch.topk(logits, k)
    threshold = top_k_vals[-1]                  # k-th largest value
    return logits.masked_fill(logits < threshold, float('-inf'))


In [ ]:
# Visualise top-k filtering
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, k in zip(axes, [5, 50, 200]):
    filtered = apply_top_k(logits, k)
    probs    = F.softmax(filtered, dim=-1)
    top_vals, top_idx = torch.topk(probs, min(k, 15))
    labels   = [repr(tokenizer.decode([i])) for i in top_idx]

    ax.bar(range(len(labels)), top_vals.cpu().numpy(), color='mediumseagreen')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Top-k (k={k})', fontweight='bold')
    ax.set_ylabel('Probability')

plt.suptitle(f'Top-k filtering on: "{prompt}"', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
def topk_decode(prompt, max_new_tokens=80, temperature=1.0, k=50, verbose=True):
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    if verbose:
        print(prompt, end='', flush=True)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits    = model(input_ids).logits[0, -1, :]
            logits    = apply_temperature(logits, temperature)
            logits    = apply_top_k(logits, k)
            probs     = F.softmax(logits, dim=-1)
            next_tok  = torch.multinomial(probs, 1).unsqueeze(0)
            input_ids = torch.cat([input_ids, next_tok], dim=-1)

            if verbose:
                print(tokenizer.decode([next_tok.item()]), end='', flush=True)

    if verbose:
        print()
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


In [ ]:
print('--- Top-k (k=10, T=0.8) ---')
topk_decode(prompt2, max_new_tokens=80, temperature=0.8, k=10)
print('\n--- Top-k (k=50, T=1.0) ---')
topk_decode(prompt2, max_new_tokens=80, temperature=1.0, k=50)


### Limitation of Top-k

k is a **fixed number** — but the 'right' number of candidates changes with context.
Sometimes the model is very confident (1 clear choice) and k=50 lets in too much noise.
Sometimes it's genuinely uncertain across hundreds of reasonable tokens.

**Top-p** adapts the candidate set size dynamically.


---
## Part 5 — Top-p (Nucleus) Sampling

**Idea:** include the smallest set of tokens whose probabilities sum to at least $p$.

Sort tokens by probability (descending), walk down until cumulative sum ≥ p.
That dynamic set is the *nucleus*.

- When the model is **confident**, the nucleus is small (a few tokens cover 90% probability).
- When the model is **uncertain**, the nucleus grows to include many reasonable options.

Introduced by Holtzman et al., 2020 — still one of the most widely used strategies.


In [ ]:
def apply_top_p(logits, p=0.9):
    """Keep only the smallest set of tokens whose cumulative probability ≥ p."""
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    probs = F.softmax(sorted_logits, dim=-1)
    cumsum = torch.cumsum(probs, dim=-1)

    # Remove tokens once cumulative prob exceeds p
    # Shift by 1 so we always keep at least the top token
    remove = cumsum - probs > p
    sorted_logits[remove] = float('-inf')

    # Restore original token order
    out = torch.full_like(logits, float('-inf'))
    out.scatter_(0, sorted_indices, sorted_logits)
    return out


In [ ]:
# How many tokens are in the nucleus for different p?
probs_full = F.softmax(logits, dim=-1)
sorted_probs, _ = torch.sort(probs_full, descending=True)
cumsum = torch.cumsum(sorted_probs, dim=-1)

print('Nucleus sizes for different p values:')
for p in [0.5, 0.7, 0.9, 0.95, 0.99]:
    n = (cumsum <= p).sum().item() + 1
    print(f'  p = {p:.2f}  →  nucleus size = {n:5d} tokens  '
          f'  (top prob = {sorted_probs[0].item():.3f})')


In [ ]:
def topp_decode(prompt, max_new_tokens=80, temperature=1.0, p=0.9, verbose=True):
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    if verbose:
        print(prompt, end='', flush=True)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits    = model(input_ids).logits[0, -1, :]
            logits    = apply_temperature(logits, temperature)
            logits    = apply_top_p(logits, p)
            probs     = F.softmax(logits, dim=-1)
            next_tok  = torch.multinomial(probs, 1).unsqueeze(0)
            input_ids = torch.cat([input_ids, next_tok], dim=-1)

            if verbose:
                print(tokenizer.decode([next_tok.item()]), end='', flush=True)

    if verbose:
        print()
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


In [ ]:
print('--- Top-p (p=0.9, T=0.8) ---')
topp_decode(prompt2, max_new_tokens=80, temperature=0.8, p=0.9)
print('\n--- Top-p (p=0.95, T=1.0) ---')
topp_decode(prompt2, max_new_tokens=80, temperature=1.0, p=0.95)


---
## Part 6 — Min-p Sampling

A newer strategy (Nguyen et al., 2024): instead of a fixed probability mass cutoff,
set a **threshold relative to the top token's probability**.

$$\text{keep token } w \quad \text{if} \quad P(w) \geq p_{\min} \cdot P(w^*)$$

where $w^*$ is the argmax token and $p_{\min} \in (0, 1)$ is a hyperparameter (e.g. 0.05).

**Why is this nice?**
- When the model is confident ($P(w^*)$ is high), the bar is high → small candidate set.
- When the model is uncertain ($P(w^*)$ is low), the bar is low → more candidates allowed.
- Automatically adaptive — like top-p, but scales with *local* confidence.

Min-p is now available in many inference frameworks (llama.cpp, HuggingFace, vLLM).


In [ ]:
def apply_min_p(logits, p_min=0.05):
    """Keep only tokens whose probability >= p_min * max_token_probability."""
    probs = F.softmax(logits, dim=-1)
    top_prob = probs.max().item()               # probability of the most likely token
    threshold = p_min * top_prob
    return logits.masked_fill(probs < threshold, float('-inf'))


In [ ]:
# Compare nucleus sizes: top-p vs min-p
probs_full = F.softmax(logits, dim=-1)
top_prob   = probs_full.max().item()

print(f'Max token prob : {top_prob:.4f}')
print()
print('Top-p nucleus sizes:')
for p in [0.9, 0.95]:
    n = (F.softmax(apply_top_p(logits, p), dim=-1) > 0).sum().item()
    print(f'  p={p}  → {n} tokens')

print('\nMin-p nucleus sizes:')
for pm in [0.1, 0.05, 0.02]:
    n = (F.softmax(apply_min_p(logits, pm), dim=-1) > 0).sum().item()
    print(f'  p_min={pm}  → threshold={pm*top_prob:.4f}  → {n} tokens')


In [ ]:
def minp_decode(prompt, max_new_tokens=80, temperature=1.0, p_min=0.05, verbose=True):
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    if verbose:
        print(prompt, end='', flush=True)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits    = model(input_ids).logits[0, -1, :]
            logits    = apply_temperature(logits, temperature)
            logits    = apply_min_p(logits, p_min)
            probs     = F.softmax(logits, dim=-1)
            next_tok  = torch.multinomial(probs, 1).unsqueeze(0)
            input_ids = torch.cat([input_ids, next_tok], dim=-1)

            if verbose:
                print(tokenizer.decode([next_tok.item()]), end='', flush=True)

    if verbose:
        print()
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


In [ ]:
print('--- Min-p (p_min=0.05, T=0.8) ---')
minp_decode(prompt2, max_new_tokens=80, temperature=0.8, p_min=0.05)
print('\n--- Min-p (p_min=0.1, T=1.0) ---')
minp_decode(prompt2, max_new_tokens=80, temperature=1.0, p_min=0.1)


---
## Part 7 — Unified Generate Function

In practice, these filters are often **combined** (temperature + top-k + top-p is common).
Let's build one function that supports all of them.


In [ ]:
def generate(
    prompt,
    max_new_tokens = 100,
    temperature    = 1.0,
    top_k          = 0,      # 0 = disabled
    top_p          = 1.0,    # 1.0 = disabled
    min_p          = 0.0,    # 0.0 = disabled
    greedy         = False,
    verbose        = True,
):
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    if verbose:
        print(prompt, end='', flush=True)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(input_ids).logits[0, -1, :]

            if greedy:
                next_tok = torch.argmax(logits).unsqueeze(0).unsqueeze(0)
            else:
                logits = apply_temperature(logits, temperature)
                if top_k  > 0:   logits = apply_top_k(logits, top_k)
                if top_p  < 1.0: logits = apply_top_p(logits, top_p)
                if min_p  > 0.0: logits = apply_min_p(logits, min_p)
                probs    = F.softmax(logits, dim=-1)
                next_tok = torch.multinomial(probs, 1).unsqueeze(0)

            input_ids = torch.cat([input_ids, next_tok], dim=-1)

            if verbose:
                print(tokenizer.decode([next_tok.item()]), end='', flush=True)

    if verbose:
        print()
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


In [ ]:
prompt3 = 'Once upon a time, in a land far away,'

configs = [
    dict(greedy=True,  label='Greedy'),
    dict(temperature=0.7, top_p=0.9,  label='T=0.7, top-p=0.9'),
    dict(temperature=0.8, top_k=50,   label='T=0.8, top-k=50'),
    dict(temperature=0.8, min_p=0.05, label='T=0.8, min-p=0.05'),
]

for cfg in configs:
    label = cfg.pop('label')
    print(f'\n=== {label} ===')
    generate(prompt3, max_new_tokens=60, verbose=True, **cfg)


---
## Part 8 — The Autoregressive Speed Bottleneck

Every token requires a **full forward pass** through the model.
These passes are **sequential** — token $t+1$ can only start after token $t$ is done.

Even if we have a 1000-GPU cluster, we still generate one token at a time.

Let's measure this concretely:


In [ ]:
def benchmark_generation(prompt, n_tokens=50):
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    token_times = []

    with torch.no_grad():
        for _ in range(n_tokens):
            t0        = time.time()
            logits    = model(input_ids).logits[0, -1, :]
            next_tok  = torch.argmax(logits).unsqueeze(0).unsqueeze(0)
            input_ids = torch.cat([input_ids, next_tok], dim=-1)
            token_times.append((time.time() - t0) * 1000)

    avg = np.mean(token_times)
    print(f'Tokens generated : {n_tokens}')
    print(f'Avg time/token   : {avg:.1f} ms')
    print(f'Throughput       : {1000/avg:.1f} tokens/sec')
    return token_times

times = benchmark_generation('The key insight of this research is', n_tokens=40)


In [ ]:
# Time grows with sequence length (KV cache not enabled here for clarity)
plt.figure(figsize=(8, 3))
plt.plot(times, 'o-', color='tomato', markersize=4)
plt.xlabel('Token step')
plt.ylabel('Time (ms)')
plt.title('Time per token — autoregressive generation is sequential')
plt.tight_layout()
plt.show()

print('\nKey insight: tokens cannot be generated in parallel.')
print('If GPT-4 has 1T parameters, it spends 1T ops for EACH token.')
print('This is the bottleneck that speculative decoding tries to solve.')


---
## Part 9 — Prompt Lookup Decoding

**The autoregressive bottleneck:** every token needs a full forward pass.
Can we generate *multiple tokens per pass* — without loading a second model?

**Key insight:** in summarisation and Q&A the output often copies phrases directly from the input.
We can exploit this: *search the prompt for what comes next*.

### Algorithm
1. Take the last few generated tokens as a search key
2. Find that n-gram somewhere in the prompt
3. Propose the tokens that follow it as a **draft**
4. Run **one** forward pass to verify them all at once
5. Accept the tokens the model agrees with — reject the rest

No second model. No extra memory. Just string search + one verify pass.


In [ ]:
def find_draft(prompt_ids, current_ids, ngram_size=2, max_draft=5):
    """
    Look for the last ngram_size tokens of current_ids inside prompt_ids.
    If found, return up to max_draft tokens that follow in the prompt.
    """
    key          = current_ids[-ngram_size:].tolist()
    prompt_list  = prompt_ids.tolist()

    for i in range(len(prompt_list) - ngram_size):
        if prompt_list[i : i + ngram_size] == key:
            return prompt_list[i + ngram_size : i + ngram_size + max_draft]
    return []   # no match found


# ── Quick demo: what does find_draft actually do? ─────────────────────────────
demo_text   = "SpaceX launches rockets to the International Space Station"
demo_ids    = tokenizer.encode(demo_text, return_tensors='pt')[0]
# Pretend we've generated "SpaceX" and want to know what comes next
query_ids   = tokenizer.encode("SpaceX", return_tensors='pt')[0]
draft       = find_draft(demo_ids, query_ids, ngram_size=1)
print(f"Prompt : {demo_text!r}")
print(f"Query  : 'SpaceX'")
print(f"Draft  : {tokenizer.decode(draft)!r}")


In [ ]:
def pld_generate(prompt, max_new_tokens=60, ngram_size=2, max_draft=5, verbose=True):
    """
    Prompt Lookup Decoding — greedy version (same model, no extra memory).

    Each step:
      1. find_draft() — search prompt for an n-gram match
      2. If found, verify all draft tokens in ONE forward pass
      3. Accept tokens where model agrees (greedy); stop at first mismatch
      4. If no match found, fall back to normal greedy for one token
    """
    prompt_ids  = tokenizer.encode(prompt, return_tensors='pt').to(device)[0]
    input_ids   = prompt_ids.unsqueeze(0)          # [1, seq_len]
    n_generated = 0
    n_drafted   = 0
    n_accepted  = 0

    if verbose:
        print(prompt, end='', flush=True)

    while n_generated < max_new_tokens:

        # ── Step 1: try to find a draft from the prompt ───────────────
        draft = find_draft(prompt_ids, input_ids[0], ngram_size, max_draft)

        if not draft:
            # No match — greedy decode one token normally
            with torch.no_grad():
                logits    = model(input_ids).logits[0, -1]
            next_tok  = logits.argmax().unsqueeze(0).unsqueeze(0)
            input_ids = torch.cat([input_ids, next_tok], dim=1)
            n_generated += 1
            if verbose:
                print(tokenizer.decode([next_tok.item()]), end='', flush=True)
            continue

        n_drafted += len(draft)

        # ── Step 2: verify the whole draft in ONE forward pass ────────
        draft_t   = torch.tensor([draft], device=device)
        cand_ids  = torch.cat([input_ids, draft_t], dim=1)  # prompt + draft

        with torch.no_grad():
            # logits at positions input_ids[-1], draft[0], draft[1], ...
            all_logits = model(cand_ids).logits[0]

        # ── Step 3: accept tokens the model agrees with ───────────────
        tokens_to_add = []
        for j, tok in enumerate(draft):
            pos         = input_ids.shape[1] - 1 + j   # which logit to check
            model_pick  = all_logits[pos].argmax().item()
            if model_pick == tok:
                tokens_to_add.append(tok)
                n_accepted += 1
            else:
                tokens_to_add.append(model_pick)        # take model's choice
                break                                   # stop at first mismatch

        # Trim to max_new_tokens budget
        tokens_to_add = tokens_to_add[:max_new_tokens - n_generated]

        add_t     = torch.tensor([tokens_to_add], device=device)
        input_ids = torch.cat([input_ids, add_t], dim=1)
        n_generated += len(tokens_to_add)

        if verbose:
            print(tokenizer.decode(tokens_to_add), end='', flush=True)

    if verbose:
        print()
        if n_drafted:
            rate = n_accepted / n_drafted
            print(f"\nDraft tokens: {n_drafted} | Accepted: {n_accepted} ({rate:.0%})")

    return input_ids


In [ ]:
# PLD works best when the output re-uses phrases from the input.
# A summarisation prompt is perfect: the summary will echo the article.

article_prompt = """Article: Elon Musk founded SpaceX in 2002 with the goal of colonizing Mars and reducing the cost of space travel. SpaceX developed the Falcon 9 rocket — the first reusable orbital rocket — and the Dragon spacecraft, which carries cargo and crew to the International Space Station. Musk also leads Tesla, the electric vehicle company, and acquired Twitter, which he later renamed X.

Q When did he found SpaceX? A:"""

print("=== Prompt Lookup Decoding demo ===\n")
pld_generate(article_prompt, max_new_tokens=60, ngram_size=2, max_draft=5)


In [ ]:
import time

def timed_greedy(prompt, max_new_tokens=60):
    ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    t0  = time.time()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            ids = torch.cat([ids, model(ids).logits[0, -1].argmax().view(1,1)], dim=1)
    elapsed = time.time() - t0
    return max_new_tokens / elapsed          # tokens per second

def timed_pld(prompt, max_new_tokens=60):
    t0  = time.time()
    pld_generate(prompt, max_new_tokens=max_new_tokens, verbose=False)
    return max_new_tokens / (time.time() - t0)

# Run both on the same prompt
print("Timing greedy vs PLD on summarisation task...\n")
greedy_tps = timed_greedy(article_prompt, max_new_tokens=60)
pld_tps    = timed_pld   (article_prompt, max_new_tokens=60)

print(f"  Greedy : {greedy_tps:.1f} tok/s")
print(f"  PLD    : {pld_tps:.1f} tok/s")
print(f"  Speedup: {pld_tps/greedy_tps:.2f}×")
print()
print("PLD skips most forward passes by reusing phrases from the prompt.")
print("The more the output echoes the input, the bigger the speedup.")


---
## Summary

| Strategy | Deterministic | Avoids repetition | Speed | Notes |
|---|---|---|---|---|
| **Greedy** | ✓ | ✗ | 1× baseline | always picks highest-prob token |
| **Temperature** | ✗ | ✓ (partial) | 1× | reshapes distribution |
| **Top-k** | ✗ | ✓ | 1× | fixed candidate set |
| **Top-p** | ✗ | ✓ | 1× | adaptive candidate set |
| **Min-p** | ✗ | ✓ | 1× | relative threshold, better calibrated |
| **PLD** | ✓ | — | **1.5–3× faster** | no second model needed |

All of the above **accept the autoregressive constraint** — one token at a time.

**Next lecture (Apr 7):** What if we removed the constraint entirely?
→ **Discrete Diffusion Language Models**
